# 01. Thống kê và trực quan hóa cơ bản

## Đề tài: Phân tích và trực quan hóa dữ liệu du lịch Việt Nam

**Mục tiêu của notebook**
- Khảo sát cấu trúc và chất lượng dữ liệu.
- Thực hiện thống kê mô tả.
- Làm sạch các giá trị đặc biệt như `..`.
- Phân tích xu hướng du lịch theo thời gian.
- So sánh khách nội địa và khách quốc tế khi dữ liệu cho phép.
- So sánh doanh thu du lịch giữa các địa phương.
- Xây dựng các biểu đồ cơ bản bằng Matplotlib/Seaborn.
- Tạo ít nhất một biểu đồ tương tác bằng Plotly.

**Nguồn dữ liệu:** dữ liệu du lịch Việt Nam do nhóm tổng hợp, gồm dữ liệu theo năm và dữ liệu theo địa phương.


In [ ]:
# 1. Import thư viện
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plotly dùng cho trực quan tương tác
import plotly.express as px

# Thiết lập hiển thị
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
# 2. Đọc dữ liệu
file_nam = "du_lieu_du_lich_nam.csv"
file_dia_phuong = "du_lieu_du_lich_dia_phuong.csv"

df_nam = pd.read_csv(file_nam)
df_dia_phuong = pd.read_csv(file_dia_phuong)

print("Kích thước dữ liệu theo năm:", df_nam.shape)
print("Kích thước dữ liệu địa phương:", df_dia_phuong.shape)


## 3. Khảo sát tổng quan dữ liệu

Trước khi phân tích, nhóm kiểm tra số dòng, số cột, một số bản ghi đầu tiên, kiểu dữ liệu và các giá trị thiếu. Đây là bước cần thiết để phát hiện các vấn đề dữ liệu trước khi trực quan hóa.


In [ ]:
# 3.1. Xem 5 dòng đầu
display(df_nam.head())
display(df_dia_phuong.head())


In [ ]:
# 3.2. Kích thước dữ liệu
print("Dataset theo năm:")
print(f"- Số dòng: {df_nam.shape[0]}")
print(f"- Số cột: {df_nam.shape[1]}")

print("\nDataset địa phương:")
print(f"- Số dòng: {df_dia_phuong.shape[0]}")
print(f"- Số cột: {df_dia_phuong.shape[1]}")


In [ ]:
# 3.3. Thông tin kiểu dữ liệu
print("=== DATASET THEO NĂM ===")
df_nam.info()

print("\n=== DATASET ĐỊA PHƯƠNG ===")
df_dia_phuong.info()


In [ ]:
# 3.4. Kiểm tra giá trị null
print("Giá trị null - dataset theo năm:")
display(df_nam.isnull().sum().to_frame("Số lượng"))

print("Giá trị null - dataset địa phương:")
display(df_dia_phuong.isnull().sum().to_frame("Số lượng"))


In [ ]:
# 3.5. Kiểm tra các ký hiệu đặc biệt như '..'
def special_value_report(df):
    result = {}
    for col in df.columns:
        result[col] = int((df[col].astype(str).str.strip() == "..").sum())
    return pd.Series(result).sort_values(ascending=False)

print("Số giá trị '..' theo cột - dataset theo năm:")
display(special_value_report(df_nam).to_frame("Số giá trị '..'"))

print("Số giá trị '..' theo cột - dataset địa phương:")
display(special_value_report(df_dia_phuong).to_frame("Số giá trị '..'"))


In [ ]:
# 3.6. Kiểm tra bản ghi trùng lặp
print("Số dòng trùng lặp - dataset theo năm:", df_nam.duplicated().sum())
print("Số dòng trùng lặp - dataset địa phương:", df_dia_phuong.duplicated().sum())


## 4. Làm sạch và chuẩn hóa dữ liệu

Trong dữ liệu nguồn, một số giá trị không có số liệu được biểu diễn bằng `..`. Ký hiệu này là chuỗi nên `isnull()` không tự động xem nó là NaN. Nhóm thay `..` bằng `NaN`, sau đó chuyển các cột có thể tính toán sang kiểu số.

Đối với dữ liệu địa phương, cột doanh thu được chuẩn hóa về kiểu numeric để có thể tính tổng, trung bình và xếp hạng.


In [ ]:
# 4.1. Thay '..' bằng NaN
df_nam_clean = df_nam.replace(r"^\s*\.\.\s*$", np.nan, regex=True).copy()
df_dia_phuong_clean = df_dia_phuong.replace(r"^\s*\.\.\s*$", np.nan, regex=True).copy()

# 4.2. Chuyển các cột có thể là số sang numeric nếu tên cột phù hợp
for df in [df_nam_clean, df_dia_phuong_clean]:
    for col in df.columns:
        if col != "Năm":
            converted = pd.to_numeric(df[col], errors="coerce")
            # Chỉ thay cột nếu việc chuyển đổi tạo ra phần lớn dữ liệu số
            non_null_original = df[col].notna().sum()
            non_null_converted = converted.notna().sum()
            if non_null_original > 0 and non_null_converted / non_null_original >= 0.7:
                df[col] = converted

# Chuẩn hóa năm
for df in [df_nam_clean, df_dia_phuong_clean]:
    if "Năm" in df.columns:
        df["Năm"] = pd.to_numeric(df["Năm"], errors="coerce").astype("Int64")

print("Kiểu dữ liệu sau chuẩn hóa - theo năm:")
display(df_nam_clean.dtypes.to_frame("dtype"))

print("Kiểu dữ liệu sau chuẩn hóa - địa phương:")
display(df_dia_phuong_clean.dtypes.to_frame("dtype"))


In [ ]:
# 4.3. Kiểm tra lại dữ liệu sau làm sạch
print("Null sau khi xử lý '..' - theo năm:")
display(df_nam_clean.isnull().sum().to_frame("Số lượng"))

print("Null sau khi xử lý '..' - địa phương:")
display(df_dia_phuong_clean.isnull().sum().to_frame("Số lượng"))


## 5. Thống kê mô tả

Thống kê mô tả giúp nhận biết quy mô, mức trung bình, độ phân tán và khoảng biến thiên của các chỉ tiêu.


In [ ]:
# 5.1. Thống kê mô tả dataset theo năm
display(df_nam_clean.describe(include="all").T)


In [ ]:
# 5.2. Thống kê mô tả dataset địa phương
display(df_dia_phuong_clean.describe(include="all").T)


## 6. Phân tích dữ liệu theo thời gian

Dataset theo năm có phạm vi 2015–2024. Nhóm tập trung quan sát xu hướng lượng khách và doanh thu, đồng thời chú ý giai đoạn suy giảm 2020–2021 và phục hồi từ 2022.


In [ ]:
# 6.1. Xu hướng các chỉ tiêu theo năm
print("Các cột khách:", ['Khách cơ sở lưu trú phục vụ (Nghìn lượt)', 'Khách nghỉ qua đêm (Nghìn lượt)', 'Khách trong ngày (Nghìn lượt)', 'Khách trong nước - cơ sở lưu trú (Nghìn lượt)', 'Khách quốc tế - cơ sở lưu trú (Nghìn lượt)', 'Khách cơ sở lữ hành phục vụ (Nghìn lượt)', 'Khách trong nước - cơ sở lữ hành (Nghìn lượt)', 'Khách quốc tế - cơ sở lữ hành (Nghìn lượt)', 'Khách Việt Nam đi nước ngoài (Nghìn lượt)'])
print("Các cột doanh thu:", ['Doanh thu cơ sở lưu trú (Tỷ đồng)', 'Doanh thu cơ sở lữ hành (Tỷ đồng)'])


In [ ]:
# Xu hướng: Khách cơ sở lưu trú phục vụ (Nghìn lượt)
plt.figure(figsize=(12,6))
plt.plot(df_nam_clean["Năm"], df_nam_clean["Khách cơ sở lưu trú phục vụ (Nghìn lượt)"], marker="o")
plt.title("Xu hướng Khách cơ sở lưu trú phục vụ (Nghìn lượt) theo năm")
plt.xlabel("Năm")
plt.ylabel("Khách cơ sở lưu trú phục vụ (Nghìn lượt)")
plt.xticks(df_nam_clean["Năm"].dropna().astype(int))
plt.tight_layout()
plt.show()


In [ ]:
# Xu hướng: Khách nghỉ qua đêm (Nghìn lượt)
plt.figure(figsize=(12,6))
plt.plot(df_nam_clean["Năm"], df_nam_clean["Khách nghỉ qua đêm (Nghìn lượt)"], marker="o")
plt.title("Xu hướng Khách nghỉ qua đêm (Nghìn lượt) theo năm")
plt.xlabel("Năm")
plt.ylabel("Khách nghỉ qua đêm (Nghìn lượt)")
plt.xticks(df_nam_clean["Năm"].dropna().astype(int))
plt.tight_layout()
plt.show()


In [ ]:
# Xu hướng: Khách trong ngày (Nghìn lượt)
plt.figure(figsize=(12,6))
plt.plot(df_nam_clean["Năm"], df_nam_clean["Khách trong ngày (Nghìn lượt)"], marker="o")
plt.title("Xu hướng Khách trong ngày (Nghìn lượt) theo năm")
plt.xlabel("Năm")
plt.ylabel("Khách trong ngày (Nghìn lượt)")
plt.xticks(df_nam_clean["Năm"].dropna().astype(int))
plt.tight_layout()
plt.show()


In [ ]:
# Xu hướng doanh thu: Doanh thu cơ sở lưu trú (Tỷ đồng)
plt.figure(figsize=(12,6))
plt.plot(df_nam_clean["Năm"], df_nam_clean["Doanh thu cơ sở lưu trú (Tỷ đồng)"], marker="o")
plt.title("Xu hướng Doanh thu cơ sở lưu trú (Tỷ đồng) theo năm")
plt.xlabel("Năm")
plt.ylabel("Doanh thu cơ sở lưu trú (Tỷ đồng) (tỷ đồng)")
plt.xticks(df_nam_clean["Năm"].dropna().astype(int))
plt.tight_layout()
plt.show()


In [ ]:
# Xu hướng doanh thu: Doanh thu cơ sở lữ hành (Tỷ đồng)
plt.figure(figsize=(12,6))
plt.plot(df_nam_clean["Năm"], df_nam_clean["Doanh thu cơ sở lữ hành (Tỷ đồng)"], marker="o")
plt.title("Xu hướng Doanh thu cơ sở lữ hành (Tỷ đồng) theo năm")
plt.xlabel("Năm")
plt.ylabel("Doanh thu cơ sở lữ hành (Tỷ đồng) (tỷ đồng)")
plt.xticks(df_nam_clean["Năm"].dropna().astype(int))
plt.tight_layout()
plt.show()


In [ ]:
# 6.2. So sánh khách trong nước và khách quốc tế
plot_df = df_nam_clean[["Năm", "Khách trong nước - cơ sở lưu trú (Nghìn lượt)", "Khách quốc tế - cơ sở lưu trú (Nghìn lượt)"]].copy()
plot_df = plot_df.melt(
    id_vars="Năm",
    var_name="Nhóm khách",
    value_name="Lượt khách"
)

plt.figure(figsize=(12,6))
sns.lineplot(data=plot_df, x="Năm", y="Lượt khách", hue="Nhóm khách", marker="o")
plt.title("Khách trong nước và khách quốc tế theo năm")
plt.xlabel("Năm")
plt.ylabel("Lượt khách (nghìn lượt)")
plt.tight_layout()
plt.show()


### Nhận xét chính

- Hoạt động du lịch có xu hướng tăng trong giai đoạn trước năm 2020.
- Giai đoạn 2020–2021 ghi nhận mức suy giảm mạnh, phù hợp với bối cảnh COVID-19.
- Từ năm 2022, các chỉ tiêu khách và doanh thu có xu hướng phục hồi.
- Khách trong nước thường chiếm quy mô lớn hơn khách quốc tế.
- Khách quốc tế có mức biến động mạnh hơn và phục hồi rõ rệt sau năm 2022.


## 7. Phân tích doanh thu theo địa phương

Dataset địa phương gồm nhiều địa phương qua nhiều năm. Để biểu đồ dễ đọc, nhóm tập trung trước vào Top 10 địa phương theo doanh thu ở năm gần nhất trong dữ liệu.


In [ ]:
# 7.1. Xác định năm gần nhất
latest_year = int(df_dia_phuong_clean["Năm"].dropna().max())
print("Năm gần nhất trong dữ liệu:", latest_year)

top10 = (
    df_dia_phuong_clean[df_dia_phuong_clean["Năm"] == latest_year]
    .dropna(subset=["Doanh thu du lịch lữ hành (Tỷ đồng)"])
    .sort_values("Doanh thu du lịch lữ hành (Tỷ đồng)", ascending=False)
    .head(10)
)

display(top10[["Địa phương", "Năm", "Doanh thu du lịch lữ hành (Tỷ đồng)"]])


In [ ]:
# 7.2. Biểu đồ Top 10 địa phương
plt.figure(figsize=(12,7))
sns.barplot(
    data=top10.sort_values("Doanh thu du lịch lữ hành (Tỷ đồng)", ascending=True),
    x="Doanh thu du lịch lữ hành (Tỷ đồng)",
    y="Địa phương"
)
plt.title(f"Top 10 địa phương có doanh thu du lịch lữ hành cao nhất năm {latest_year}")
plt.xlabel("Doanh thu (tỷ đồng)")
plt.ylabel("Địa phương")
plt.tight_layout()
plt.show()


In [ ]:
# 7.3. Plotly tương tác
fig = px.bar(
    top10.sort_values("Doanh thu du lịch lữ hành (Tỷ đồng)", ascending=True),
    x="Doanh thu du lịch lữ hành (Tỷ đồng)",
    y="Địa phương",
    orientation="h",
    title=f"Top 10 địa phương có doanh thu du lịch lữ hành cao nhất năm {latest_year}",
    labels={
        "Doanh thu du lịch lữ hành (Tỷ đồng)": "Doanh thu (tỷ đồng)",
        "Địa phương": "Địa phương"
    },
    hover_data=["Năm"]
)
fig.show()


### Nhận xét

Biểu đồ Top 10 giúp làm nổi bật sự khác biệt về doanh thu giữa các địa phương. Các trung tâm du lịch và đô thị lớn có xu hướng nằm trong nhóm dẫn đầu. Việc sử dụng biểu đồ cột ngang giúp đọc tên địa phương thuận tiện hơn khi số lượng nhóm tương đối nhiều.

Với Plotly, người xem có thể di chuột để xem giá trị cụ thể, giúp tăng khả năng khai thác dữ liệu so với biểu đồ tĩnh.


## 8. Phân tích sâu hơn: doanh thu theo thời gian của một số địa phương

Ngoài việc xếp hạng tại một năm, có thể quan sát sự thay đổi của các địa phương nổi bật qua toàn bộ giai đoạn.


In [ ]:
# 8.1. Chọn Top 5 địa phương theo doanh thu ở năm gần nhất
top5_places = top10.head(5)["Địa phương"].tolist()
trend_dp = df_dia_phuong_clean[
    df_dia_phuong_clean["Địa phương"].isin(top5_places)
].dropna(subset=["Doanh thu du lịch lữ hành (Tỷ đồng)"])

display(trend_dp.head())


In [ ]:
# 8.2. Biểu đồ xu hướng Top 5 địa phương
plt.figure(figsize=(13,7))
sns.lineplot(
    data=trend_dp,
    x="Năm",
    y="Doanh thu du lịch lữ hành (Tỷ đồng)",
    hue="Địa phương",
    marker="o"
)
plt.title("Xu hướng doanh thu du lịch lữ hành của Top 5 địa phương")
plt.xlabel("Năm")
plt.ylabel("Doanh thu (tỷ đồng)")
plt.tight_layout()
plt.show()


## 9. Tổng hợp các phát hiện chính

1. Dữ liệu du lịch theo năm cho phép quan sát rõ xu hướng tăng trưởng, suy giảm và phục hồi của ngành.
2. Giai đoạn 2020–2021 là điểm biến động lớn nhất trong chuỗi thời gian.
3. Từ năm 2022, hoạt động du lịch phục hồi rõ rệt.
4. Khách trong nước có quy mô lớn, trong khi khách quốc tế biến động mạnh hơn.
5. Doanh thu du lịch có sự khác biệt đáng kể giữa các địa phương.
6. Một số trung tâm du lịch lớn nổi bật về doanh thu.
7. Biểu đồ tĩnh phù hợp với các phân tích tổng quan; Plotly phù hợp khi cần khám phá chi tiết từng địa phương hoặc từng năm.


## 10. Kết luận phần Notebook 01

Phần Notebook 01 đã hoàn thành quy trình cơ bản gồm đọc dữ liệu, kiểm tra chất lượng, xử lý dữ liệu, thống kê mô tả và trực quan hóa. Kết quả giúp nhóm hình thành cái nhìn tổng quan về tình hình du lịch Việt Nam giai đoạn 2015–2024 và tạo nền tảng cho các phân tích chuyên sâu hơn ở các phần tiếp theo.

> **Lưu ý:** Không nên kết luận quan hệ nhân quả chỉ từ biểu đồ. Ví dụ, nếu lượng khách và doanh thu cùng tăng thì có thể nói chúng có xu hướng cùng chiều trong dữ liệu, nhưng chưa đủ cơ sở để khẳng định lượng khách là nguyên nhân trực tiếp làm doanh thu tăng.
